# 第 13 章 IS-LM 分析: 数字を変えて確かめる

**使い方**: 「Parameters」のセルの数字を書き換えて、上から順に実行します（メニューの Run → Run All Cells）。数字を変えるたびに、結果と図が変わります。

起動: アプリのディレクトリで `uv run --with jupyterlab jupyter lab notebooks/` を実行します。

記事: `docs/article/economics-programming/13-is-lm-analysis.md`

In [ ]:
# --- Parameters: ここの数字を書き換えます ---
autonomous_consumption = 100  # C0: 基礎消費
mpc = 0.8                     # c: 限界消費性向
tax = 50                      # T: 定額税
autonomous_investment = 140   # I0: 利子率が 0 のときの投資
investment_sensitivity = 10   # b: 投資の利子率感応度
government = 100              # G: 政府支出
money_supply = 525            # M: 名目貨幣供給
price_level = 1               # P: 物価水準
money_income_sensitivity = 0.5      # k: 貨幣需要の所得感応度
money_interest_sensitivity = 20     # h: 貨幣需要の利子率感応度
interest_floor = None         # 利子率の下限（流動性のわな）。例: 2
government_after = 100        # 政策後の政府支出（財政政策）
money_supply_after = 525      # 政策後の貨幣供給（金融政策）

In [ ]:
# --- 準備（変更しません）: lib を読み込めるようにします ---
import sys
from pathlib import Path

_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "lib").is_dir())
if str(_root) in sys.path:
    sys.path.remove(str(_root))
sys.path.insert(0, str(_root))
_lib = sys.modules.get("lib")
if _lib is not None and (
    Path(getattr(_lib, "__file__", "") or "").resolve().parent
    != _root / "lib"
):
    for _name in list(sys.modules):
        if _name == "lib" or _name.startswith("lib."):
            del sys.modules[_name]

## IS-LM の均衡と、政策の効果

In [ ]:
import dataclasses

from lib.chapter13.islm import IsLmModel, solve_equilibrium

model = IsLmModel(
    autonomous_consumption=autonomous_consumption,
    mpc=mpc,
    tax=tax,
    autonomous_investment=autonomous_investment,
    investment_sensitivity=investment_sensitivity,
    government=government,
    money_supply=money_supply,
    price_level=price_level,
    money_income_sensitivity=money_income_sensitivity,
    money_interest_sensitivity=money_interest_sensitivity,
    interest_floor=interest_floor,
)
after = dataclasses.replace(
    model, government=government_after, money_supply=money_supply_after
)
income, rate = solve_equilibrium(model)
income_after, rate_after = solve_equilibrium(after)
print(f"政策前: 所得 {income:.1f}、利子率 {rate:.2f}")
print(f"政策後: 所得 {income_after:.1f}、利子率 {rate_after:.2f}")

## 図: IS 曲線と LM 曲線

In [ ]:
from lib.chapter13.islm import plot_islm

plot_islm(model, after)

## 試してみよう

- `government_after` を 150 にすると、所得は 1250 から 1361.1 に、利子率は 5.00 から
  7.78 に上がります（クラウディングアウト: 利子率が上がって、投資が減ります）
- `money_supply_after` を 625 にすると、所得は 1361.1 に増え、利子率は 2.78 に
  下がります
- `interest_floor` を 2 にして、`money_supply_after` を 2000 にすると、利子率は
  下限の 2.00 に張り付き、所得は 1400 で止まります。3000 にしても、1400 のままです
  （流動性のわな: 金融政策が効きません）
- `interest_floor` を 2、`money_supply` と `money_supply_after` を 2000 にして、
  `government_after` を 120 にすると、所得は 100 増えます（クラウディングアウトが
  なく、乗数どおり）